In [2]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything

# Get inference device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Init model - This requries internet access or the huggingface hub cache to be pre-downloaded
# For Apache 2.0 license model, use "facebook/map-anything-apache"
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


In [3]:
from pathlib import Path
from PIL import Image
import numpy as np
import torch

from mapanything.models import MapAnything
from mapanything.utils.image import preprocess_inputs

BASE = Path("/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3")
ALL_RGB = sorted((BASE / "linearPNG").glob("*.png"))

# Pick frames 0, 120, 240 (0-based indexing)
frame_idxs = [0]
selected_rgb = [ALL_RGB[i] for i in frame_idxs if i < len(ALL_RGB)]

# K = np.array([[987.46, 0.0, 830.36],
#               [0.0, 987.46, 644.75],
#               [0.0, 0.0, 1.0]], dtype=np.float32)

K = np.array(
    [
        [1523.27, 0.0, 798.97],
        [0.0, 1523.27, 646.51],
        [0.0,    0.0,   1.0 ],
    ],
    dtype=np.float32,
)


views = []
for rgb_path in selected_rgb:
    depth_path = BASE / "decoded_npy" / f"{rgb_path.stem}.npy"
    if not depth_path.exists():
        continue

    rgb = np.array(Image.open(rgb_path).convert("RGB"))
    depth = np.load(depth_path).astype(np.float32)

    views.append({
        "img": rgb,
        "intrinsics": K,
        "depth_z": depth,
        "is_metric_scale": torch.tensor([True]),
    })

processed_views = preprocess_inputs(views)


In [4]:
rgb.shape

(300, 400, 3)

In [3]:
# Run inference with any combination of inputs
predictions = model.infer(
    processed_views,                  # Any combination of input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=True,                  # Apply masking to dense geometry outputs
    mask_edges=True,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
    # Control which inputs to use/ignore
    # By default, all inputs are used when provided
    # If is_metric_scale flag is not provided, all inputs are assumed to be in metric scale
    ignore_calibration_inputs=False,
    ignore_depth_inputs=False,
    ignore_pose_inputs=False,
    ignore_depth_scale_inputs=False,
    ignore_pose_scale_inputs=False,
)

In [27]:
print(predictions)

[{'pts3d': tensor([[[[-10.5989,  -8.0272,  15.9938],
          [-10.5887,  -8.1568,  16.1337],
          [-10.6167,  -8.1570,  15.9311],
          ...,
          [  8.1974,  -6.4232,  13.6978],
          [  0.0000,  -0.0000,   0.0000],
          [  7.9806,  -6.1262,  13.2768]],

         [[-10.6555,  -8.1185,  15.7886],
          [-10.5749,  -8.0990,  16.1651],
          [-10.6811,  -8.1213,  15.8912],
          ...,
          [  8.2005,  -6.3908,  13.6120],
          [  0.0000,  -0.0000,   0.0000],
          [  8.2080,  -6.2826,  13.4910]],

         [[-10.8720,  -8.1989,  15.9156],
          [-10.6599,  -8.1028,  16.0577],
          [-10.6350,  -8.0739,  16.0177],
          ...,
          [  8.2395,  -6.4175,  13.7780],
          [  8.2560,  -6.4228,  13.6486],
          [  8.2243,  -6.3033,  13.6550]],

         ...,

         [[-35.0381,  25.4665,  54.0987],
          [-34.5559,  24.6949,  54.0953],
          [-34.4005,  24.8899,  54.8388],
          ...,
          [ 16.8387,  12.9

In [4]:
import numpy as np
import open3d as o3d

pred = predictions[0]  # first view

pts3d = pred["pts3d"].squeeze().cpu().numpy()            # (H, W, 3)
colors = pred["img_no_norm"].squeeze().cpu().numpy()     # (H, W, 3) in [0, 1]
mask = pred["mask"].squeeze().cpu().numpy().astype(bool) # validity mask

xyz_cam = pts3d[mask]
rgb = colors[mask]

# # Camera frame (OpenCV) -> world frame (X right, Y forward, Z up)
# R_cv_to_world = np.array([
#     [1.0, 0.0,  0.0],
#     [0.0, 0.0,  1.0],
#     [0.0, -1.0, 0.0],
# ], dtype=np.float32)


# xyz_world = xyz_cam @ R_cv_to_world.T


# pcd = o3d.geometry.PointCloud()
# pcd.points = o3d.utility.Vector3dVector(xyz)
# pcd.colors = o3d.utility.Vector3dVector(rgb)

# o3d.visualization.draw_geometries([pcd], window_name="MapAnything Reconstruction")

# Flip Y to make “up” match the image; adjust as needed
R_cam_fix = np.array([[1, 0, 0],
                      [0,-1, 0],
                      [0, 0, 1]], dtype=np.float32)
xyz_cam_aligned = xyz_cam @ R_cam_fix.T

# If you need them in world space again:
T_cam0 = pred["camera_poses"][0].cpu().numpy()   # cam→world
xyz_world = xyz_cam_aligned @ T_cam0[:3, :3].T + T_cam0[:3, 3]

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(xyz_world)
pcd.colors = o3d.utility.Vector3dVector(rgb)
o3d.visualization.draw_geometries([pcd])


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [ ]:
import time
from collections import deque
from pathlib import Path

import numpy as np
import open3d as o3d
import torch
from PIL import Image

from mapanything.models import MapAnything
from mapanything.utils.image import preprocess_inputs

# --------------------------------------------------------------------
# Data locations and camera intrinsics
# --------------------------------------------------------------------
BASE = Path(
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/"
    "scrippsDivesWithDepth/depth3"
)
RGB_DIR = BASE / "linearPNG"
DEPTH_DIR = BASE / "decoded_npy"

K = np.array(
    [
        [987.46, 0.0, 830.36],
        [0.0, 987.46, 644.75],
        [0.0, 0.0, 1.0],
    ],
    dtype=np.float32,
)

FRAME_IDS = sorted(RGB_DIR.glob("*.png"))
FRAME_SKIP = 60             # advance ~1 s @ 60 fps
WINDOW_SIZE = 3             # keep last 3 frames

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MapAnything.from_pretrained("facebook/map-anything").to(device).eval()

# --------------------------------------------------------------------
# Open3D viewer setup
# --------------------------------------------------------------------
pcd = o3d.geometry.PointCloud()
vis = o3d.visualization.Visualizer()
vis.create_window(window_name="Live RGB-D Map")
vis.add_geometry(pcd)

frame_window: deque[Path] = deque(maxlen=WINDOW_SIZE)

with torch.inference_mode():
    for idx in range(0, len(FRAME_IDS), FRAME_SKIP):
        rgb_path = FRAME_IDS[idx]
        depth_path = DEPTH_DIR / f"{rgb_path.stem}.npy"
        if not depth_path.exists():
            continue

        frame_window.append(rgb_path)
        # ----------------------------------------------------------------
        # Build multimodal views for the current window
        # ----------------------------------------------------------------
        views = []
        for path in frame_window:
            depth_file = DEPTH_DIR / f"{path.stem}.npy"
            if not depth_file.exists():
                continue
            rgb = np.array(Image.open(path).convert("RGB"))
            depth = np.load(depth_file).astype(np.float32)

            views.append(
                {
                    "img": rgb,
                    "intrinsics": K,
                    "depth_z": depth,
                    "is_metric_scale": torch.tensor([True], device=device),
                }
            )

        if len(views) < 1:
            continue

        processed = preprocess_inputs(views)
        # Move tensors to the same device as the model
        processed = [
            {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in view.items()}
            for view in processed
        ]

        # ----------------------------------------------------------------
        # MapAnything inference for the current window
        # ----------------------------------------------------------------
        predictions = model.infer(processed, use_amp=True)

        xyz_all = []
        rgb_all = []
        for pred in predictions:
            pts3d_world = pred["pts3d"].squeeze().cpu().numpy()
            img_rgb = pred["img_no_norm"].squeeze().cpu().numpy()
            mask = pred["mask"].squeeze().cpu().numpy().astype(bool)

            xyz_all.append(pts3d_world[mask])
            rgb_all.append(img_rgb[mask])

        if not xyz_all:
            continue

        xyz_live = np.concatenate(xyz_all, axis=0)
        rgb_live = np.concatenate(rgb_all, axis=0)

        # ----------------------------------------------------------------
        # Refresh Open3D with the latest window (overwrites previous data)
        # ----------------------------------------------------------------
        pcd.points = o3d.utility.Vector3dVector(xyz_live)
        pcd.colors = o3d.utility.Vector3dVector(rgb_live)
        vis.update_geometry(pcd)
        vis.poll_events()
        vis.update_renderer()

        # Simple pacing so the visualization is readable; adjust as needed
        time.sleep(0.1)

vis.destroy_window()


Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main
